## 모델의 성능 검증하기

In [17]:
# ===============================
# 1. 경고 메시지 숨기기
# ===============================
# 라이브러리 버전 차이 등으로 나오는 경고(warning)는
# 학습 자체에는 영향이 없기 때문에 화면에서 숨김
import warnings
warnings.filterwarnings('ignore')

In [18]:
# ===============================
# 2. 데이터 처리 라이브러리
# ===============================
# pandas : CSV, 엑셀 같은 표 형태 데이터를 다루는 라이브러리
import pandas as pd


# ===============================
# 3. 데이터 불러오기
# ===============================
# sonar3.csv 파일을 읽어서 df(DataFrame)에 저장
# header=None : 컬럼 이름이 없는 데이터이기 때문에
#               0, 1, 2, ... 자동 번호를 컬럼명으로 사용
df=pd.read_csv('./sonar3.csv',header=None)


# 데이터가 제대로 불러와졌는지
# 맨 위 5줄을 확인 (출력용)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,0
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,0
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,0
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,0
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,0


In [19]:
# ===============================
# 4. 입력 데이터(X) / 정답(y) 분리
# ===============================
# iloc : 행과 열을 "숫자 인덱스"로 선택


# X : 입력 데이터 (특징, feature)
# 모든 행(:)에서
# 0 ~ 59 열까지 선택 → 총 60개의 입력값
X=df.iloc[:,0:60]

# y : 정답 데이터 (라벨, target)
# 모든 행(:)에서
# 60번째 열 하나만 선택
# → 0 또는 1 (이진 분류 문제)
y=df.iloc[:,60]


In [20]:

# ===============================
# 5. 신경망 관련 라이브러리
# ===============================
# Sequential : 층을 위에서 아래로 순서대로 쌓는 모델
# Dense      : 완전 연결층 (Fully Connected Layer)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


# ===============================
# 6. 신경망 모델 생성
# ===============================
# 비어있는 신경망 틀 생성
model=Sequential()


# ===============================
# 7. 첫 번째 은닉층
# ===============================
# Dense(24, input_dim=60, activation='relu')
#
# - input_dim=60
#   → 입력 데이터 X의 열 개수
#   → 특징(feature)이 60개라는 뜻
#
# - 24
#   → 이 층의 뉴런(노드) 개수
#   → 60개의 정보를 받아서 24개의 새로운 특징으로 변환
#
# - activation='relu'
#   → ReLU 활성화 함수
#   → 음수는 0, 양수는 그대로
#   → 계산 빠르고 기울기 소실 문제 완화


model.add(Dense(24,input_dim=60,activation='relu'))

# ===============================
# 8. 두 번째 은닉층
# ===============================
# 이전 층(24개 출력)을 입력으로 받음
# 뉴런 수를 10개로 줄여서
# 더 핵심적인 특징만 남김


model.add(Dense(10,activation='relu'))


# ===============================
# 9. 세 번째 은닉층
# ===============================
# activation='tanh'
# - 출력 범위: -1 ~ 1
# - ReLU만 계속 쓰면 값이 한쪽으로 치우칠 수 있어서
#   중간에 tanh를 넣어 균형 잡힌 표현 학습


model.add(Dense(10,activation='tanh'))




# ===============================
# 10. 출력층 (가장 중요)
# ===============================
# Dense(1, activation='sigmoid')
#
# - 뉴런 1개
#   → 결과를 하나의 숫자로 출력
#
# - sigmoid
#   → 출력값을 0 ~ 1 사이 확률로 변환
#   → 0에 가까우면 클래스 0
#   → 1에 가까우면 클래스 1
#
# 이진 분류 문제의 정석적인 출력층 구조
model.add(Dense(1,activation='sigmoid'))

# ===============================
# 11. 모델 컴파일 (학습 방법 설정)
# ===============================
               # loss : 손실 함수
            # binary_crossentropy
            # → 이진 분류 문제 전용 손실 함수
            # → 예측이 틀릴수록 큰 벌점
model.compile(loss='binary_crossentropy',
              
                 # optimizer : 가중치를 어떻게 업데이트할지
                # adam
                # → 학습률을 자동으로 조절하는 가장 대중적인 알고리즘
              optimizer='adam',
              metrics=['accuracy'])

# ===============================
# 12. 모델 학습
# ===============================
history=model.fit(X, # 입력 데이터
                  y,  # 정답 데이터
                    # epochs=200
                    # → 전체 데이터를 처음부터 끝까지
                    #    200번 반복해서 학습
                  epochs=200,
                     # batch_size=10
                    # → 데이터를 10개씩 나눠서 학습
                    # → 너무 크면 메모리 부담
                    # → 너무 작으면 학습이 불안정
                  batch_size=10)


# history 변수에는
# 각 epoch마다의 loss, accuracy 기록이 저장됨
# → 그래프로 시각화 가능

Epoch 1/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5288 - loss: 0.6914   
Epoch 2/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6058 - loss: 0.6731 
Epoch 3/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5913 - loss: 0.6566 
Epoch 4/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7404 - loss: 0.6396 
Epoch 5/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6971 - loss: 0.6103 
Epoch 6/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7596 - loss: 0.5823 
Epoch 7/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7692 - loss: 0.5502 
Epoch 8/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7452 - loss: 0.5199 
Epoch 9/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7981 - loss: 0.4998 
Epoch 10/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7885 - loss: 0.4787 
Epoch 11/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8221 - loss: 0.4493 
Epoch 12/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/ste

In [21]:
# ===============================
# 1. 필요한 라이브러리 불러오기
# ===============================

# Sequential : 층을 순서대로 쌓는 신경망 모델
# Dense      : 완전 연결층 (Fully Connected Layer)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# train_test_split : 데이터를 학습용 / 테스트용으로 나누는 함수
from sklearn.model_selection import train_test_split
# pandas : CSV 파일을 표 형태(DataFrame)로 다루기 위한 라이브러리
import pandas as pd


# ===============================
# 2. 데이터 불러오기
# ===============================

# sonar3.csv 파일을 읽어와서 df(DataFrame)에 저장
# header=None : 컬럼명이 없기 때문에 자동으로 0,1,2,... 번호 부여
df=pd.read_csv('./sonar3.csv',header=None)

In [22]:
# ===============================
# 3. 입력 데이터(X) / 정답(y) 분리
# ===============================

# X : 입력 데이터 (특징, feature)
# 모든 행(:)에서 0 ~ 59 열까지 선택
# → 초음파 신호 값 60개

X=df.iloc[:,0:60]

# y : 정답 데이터 (라벨, target)
# 모든 행(:)에서 60번째 열 선택
# → 0 또는 1 (이진 분류 문제)
y=df.iloc[:,60]


# ===============================
# 4. 학습용 / 테스트용 데이터 분리
# ===============================

# train_test_split
# - X, y를 같은 비율로 나눔
# - 모델이 "처음 보는 데이터"에서도 잘 동작하는지 확인하기 위함
X_train,X_test,y_train,y_test=train_test_split(X,# 전체 입력 데이터
                                               y, # 전체 정답 데이터
                                               test_size=0.3, # 전체 데이터의 30%를 테스트용으로 사용
                                               shuffle=True) # 데이터를 섞어서 분리 (순서 편향 방지)

In [23]:
# 결과:
# X_train, y_train → 모델 학습용 (70%)
# X_test, y_test   → 성능 평가용 (30%)


# ===============================
# 5. 신경망 모델 생성
# ===============================

# 순차적으로 층을 쌓는 모델 생성
model=Sequential()

# ===============================
# 6. 첫 번째 은닉층
# ===============================

# Dense(24, input_dim=60, activation='relu')
#
# - input_dim=60
#   → 입력 데이터의 특징 개수
#   → X_train의 열 개수와 반드시 같아야 함
#
# - 24
#   → 이 층의 뉴런(노드) 개수
#   → 60개의 입력 정보를 24개의 핵심 특징으로 변환
#
# - activation='relu'
#   → ReLU 활성화 함수
#   → 음수는 0으로, 양수는 그대로 출력
#   → 계산이 빠르고 딥러닝에서 가장 많이 사용
model.add(Dense(24,input_dim=60,activation='relu'))


# ===============================
# 7. 두 번째 은닉층
# ===============================

# 이전 층의 출력(24개)을 입력으로 받음
# 뉴런 수를 10개로 줄여서
# 더 중요한 특징만 남기도록 압축
model.add(Dense(10,activation='relu'))



# ===============================
# 8. 출력층
# ===============================

# Dense(1, activation='sigmoid')
#
# - 뉴런 1개
#   → 결과를 하나의 숫자로 출력
#
# - sigmoid
#   → 출력값을 0 ~ 1 사이의 확률로 변환
#   → 0에 가까우면 클래스 0
#   → 1에 가까우면 클래스 1
#
# 이진 분류 문제에서 표준적인 출력층 구조
model.add(Dense(1,activation='sigmoid'))




# ===============================
# 9. 모델 컴파일 (학습 방법 설정)
# ===============================
model.compile(
            # loss : 손실 함수
            # binary_crossentropy
            # → 이진 분류 문제에 최적화된 손실 함수
              loss='binary_crossentropy',
               # optimizer : 가중치를 업데이트하는 방법
               # adam
               # → 학습률을 자동으로 조절하는 가장 널리 쓰이는 알고리즘
              optimizer='adam',
              # metrics : 학습 중 함께 출력할 지표
              # accuracy : 정확도
              metrics=['accuracy'])

# ===============================
# 10. 모델 학습
# ===============================
history=model.fit( # 학습용 입력 데이터
                  X_train,
                  y_train,# 학습용 정답 데이터
                   # epochs=200
                   # → 전체 학습 데이터를 200번 반복해서 학습
                  epochs=200,
                  
                   # batch_size=10
                   # → 데이터를 10개씩 나눠서 학습
                   # → 메모리 효율 + 학습 안정성
                  batch_size=10)

Epoch 1/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6000 - loss: 0.6891  
Epoch 2/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6552 - loss: 0.6723 
Epoch 3/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6759 - loss: 0.6590 
Epoch 4/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7034 - loss: 0.6467 
Epoch 5/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7517 - loss: 0.6363 
Epoch 6/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7379 - loss: 0.6244 
Epoch 7/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7448 - loss: 0.6138 
Epoch 8/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7103 - loss: 0.6080 
Epoch 9/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7310 - loss: 0.5938 
Epoch 10/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7310 - loss: 0.5869 
Epoch 11/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7517 - loss: 0.5757 
Epoch 12/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

In [ ]:
# ===============================
# 1. 테스트 데이터로 모델 평가
# ===============================

# model.evaluate()
# → 모델이 "처음 보는 데이터(X_test)"를 얼마나 잘 맞히는지 평가
score=model.evaluate(X_test,y_test)

# score에는 리스트 형태로 값이 들어 있음
# score[0] : loss (얼마나 틀렸는지)
# score[1] : accuracy (얼마나 맞았는지)
print('Test accuracy:',score[1])

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8889 - loss: 0.3907
Test accuracy: 0.8888888955116272


In [ ]:
# ===============================
# 2. 학습된 모델 저장
# ===============================

# model.save()
# → 학습된 신경망을 파일로 저장
# → 가중치 + 구조 + 설정까지 전부 저장됨
model.save('./my_model.keras')

In [ ]:
# ===============================
# 3. 저장된 모델 불러오기
# ===============================

# load_model : 저장된 모델을 다시 불러오는 함수
from tensorflow.keras.models import Sequential,load_model


In [27]:
del model

In [ ]:
# 디스크에 저장된 모델 파일을 메모리로 로드
model=load_model('./my_model.keras')

# ===============================
# 4. 불러온 모델 다시 평가
# ===============================

# 저장했다가 불러온 모델로
# 테스트 데이터를 다시 평가
score=model.evaluate(X_test,y_test)
print("Test accuracy:",score[1])

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8889 - loss: 0.3907 
Test accuracy: 0.8888888955116272


In [29]:
# ===============================
# 1. 필요한 라이브러리 불러오기
# ===============================

# Sequential : 층(layer)을 위에서 아래로 차례대로 쌓는 신경망 모델
# Dense      : 모든 노드가 서로 연결된 기본 신경망 층
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# KFold : 데이터를 여러 조각으로 나눠서
#         여러 번 학습/평가를 하게 해주는 도구
from sklearn.model_selection import KFold

# accuracy_score : 정확도 계산 함수
# (이번 코드에서는 직접 사용하지 않지만 참고용)
from sklearn.metrics import accuracy_score
# pandas : CSV 파일 같은 표 형태 데이터를 다루는 라이브러리
import pandas as pd


# ===============================
# 2. 입력 데이터(X)와 정답(y) 분리
# ===============================

# X : 입력 데이터 (문제)
# 모든 행(:)에서
# 0번 열부터 59번 열까지 → 총 60개의 특징
X=df.iloc[:,0:60]

# y : 정답 데이터 (답지)
# 모든 행(:)에서
# 60번째 열 하나 → 0 또는 1 (이진 분류)
y=df.iloc[:,60]


In [30]:
# ===============================
# 3. K-Fold 설정
# ===============================

# 데이터를 몇 조각으로 나눌지 설정

k=5


# KFold 객체 생성
# n_splits=5 : 데이터를 5등분
# shuffle=True : 데이터를 먼저 섞어서
#               특정 패턴으로 나뉘는 것을 방지
kfold=KFold(n_splits=k,shuffle=True)


# ===============================
# 4. 각 fold의 정확도를 저장할 리스트
# ===============================

# 5번 시험 본 정확도를 하나씩 저장
acc_score=[]


# ===============================
# 5. 신경망 모델을 만드는 함수
# ===============================

# 왜 함수로 만드나?
# → K-Fold에서는 매번 "완전히 새로운 모델"이 필요함
def model_fn():
      # 순차적으로 층을 쌓는 신경망 생성
    model=Sequential()

    # 첫 번째 은닉층
    # - 입력값: 60개
    # - 뉴런: 24개
    # - ReLU 활성화 함수 사용
    model.add(Dense(24,input_dim=60,activation='relu'))
    
    # 두 번째 은닉층
    # - 뉴런: 10개
    model.add(Dense(10,activation='relu'))

    # 출력층
    # - 뉴런 1개
    # - sigmoid → 0~1 사이 값 출력
    # → 이진 분류에 사용
    model.add(Dense(1,activation='sigmoid'))
    return model


# ===============================
# 6. K-Fold 교차검증 시작
# ===============================

# kfold.split(X)는 반복할 때마다
# - 학습용 데이터 번호(train_index)
# - 테스트용 데이터 번호(test_index)
# 를 자동으로 만들어 줌
for train_index,test_index in kfold.split(X):
     # 번호표(train_index, test_index)를 이용해
    # 실제 데이터를 나눔

    # 학습용 입력 데이터
    X_train,X_test=X.iloc[train_index,:],X.iloc[test_index,:]
    
    y_train,y_test=y.iloc[train_index],y.iloc[test_index]

    # ===============================
    # 7. 모델 생성 및 설정
    # ===============================

    # 새 신경망 모델 생성
    model=model_fn()
    # 모델 학습 방법 설정
    model.compile(loss='binary_crossentropy', # 이진 분류용 손실 함수
                  optimizer='adam',# 가장 많이 쓰이는 최적화 알고리즘
                  metrics=['accuracy'])# 정확도 측정
    
    # ===============================
    # 8. 모델 학습
    # ===============================

    # 학습 시작
    # epochs=200 : 전체 데이터를 200번 반복 학습
    # batch_size=10 : 한 번에 10개 데이터씩 학습
    # verbose=0 : 학습 과정 출력 안 함
    histroy=model.fit(X_train,y_train,epochs=200,batch_size=10,verbose=0)


    # ===============================
    # 9. 모델 평가
    # ===============================

    # 테스트 데이터로 성능 평가
    # evaluate 결과는 [loss, accuracy]
    accuracy=model.evaluate(X_test,y_test)[1]
    # 이번 fold의 정확도를 리스트에 저장
    acc_score.append(accuracy)

# ===============================
# 10. 평균 정확도 계산
# ===============================

# 5번의 정확도를 평균냄
avg_acc_score=sum(acc_score)/k


# ===============================
# 11. 결과 출력
# ==============================
print('정확도:',acc_score)
print("정확도 평균:",avg_acc_score)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7857 - loss: 0.9078
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8333 - loss: 0.7458
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9048 - loss: 0.5629
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8293 - loss: 0.9702
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8049 - loss: 0.6411
정확도: [0.7857142686843872, 0.8333333134651184, 0.9047619104385376, 0.8292682766914368, 0.8048780560493469]
정확도 평균: 0.8315911650657654
